# High-z Example 23: RAG-Ready JSONL Demonstration

**EPS Research — High-z Kinematic Corpus Z1**

The JSONL format (one galaxy per line) is optimized for LLM
retrieval-augmented generation (RAG) pipelines.
Each line is a self-contained JSON object with full metadata —
no external lookups needed to answer kinematic queries.

This example shows how to load and query the JSONL format.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np

# Load JSONL (one galaxy per line)
galaxies_jsonl = []
with open('high_z_kinematic_corpus_Z1.jsonl') as f:
    for line in f:
        galaxies_jsonl.append(json.loads(line))

print(f"Loaded {len(galaxies_jsonl)} galaxy records from JSONL")
print(f"\nEach record is self-contained:")
g = galaxies_jsonl[0]
print(f"  Keys: {list(g.keys())}")
print(f"\nSize per record: ~{len(json.dumps(g))/1024:.1f} KB")
print(f"  (well within LLM context limits)")
print(f"\nExample RAG query: 'What is the omega value for {g['galaxy']}?'")

# Simulate a RAG retrieval
target = 'DC552206'
record = next((g for g in galaxies_jsonl if g['galaxy'] == target),
              galaxies_jsonl[0])
d = record.get('data', [])
if d:
    R1,V1=d[0]['R_kpc'],d[0]['Vrot_kms']
    R2,V2=d[-1]['R_kpc'],d[-1]['Vrot_kms']
    outer_term    = (V2 / R2)
    inner_term    = (V1 / R1) * ((R1 / R2) ** 1.5)
    omega_kms_kpc = outer_term - inner_term          # Flynn & Cannaliato 2025 Eq.6  [km/s/kpc]
    omega_rad_gyr = omega_kms_kpc * 1.0227           # 1 km/s/kpc = 1.0227 rad/Gyr
    omega = omega_rad_gyr  # Eq.6 corrected 2026-07-12: operator-precedence fix
    print(f"\nRAG answer for {target}:")
    print(f"  omega = {omega:.3f} rad/Gyr")
    print(f"  Vmax = {record.get('vrot_max_kms','?')} km/s")
    print(f"  z = {record['redshift']}")